# Conformal Prediction (MC-Dropout-normalized)

Basic conformal prediction gives every test point the *same* interval width, which would give the accept/abstain sweep nothing to threshold on -- every engine would look equally (un)certain. This notebook uses the normalized/locally-weighted variant instead: MC Dropout's per-point std (same trick as `dropout.ipynb`) acts as a cheap per-point "difficulty" signal, and conformal calibration on `val_df` turns that signal into a statistically calibrated interval width per engine. This is the same combination the Xuan et al. paper already cited in the literature review uses.

In [6]:
import pandas as pd
import numpy as np


In [7]:
df = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\Dataset\processed_train.csv")

np.random.seed(42)

engine_ids = df["Engine_ID"].unique()
np.random.shuffle(engine_ids)

train_ids = engine_ids[:70]
val_ids = engine_ids[70:85]
test_ids = engine_ids[85:]

train_df = df[df["Engine_ID"].isin(train_ids)].copy()
val_df = df[df["Engine_ID"].isin(val_ids)].copy()
test_df = df[df["Engine_ID"].isin(test_ids)].copy()

print("Training engines:", train_df["Engine_ID"].nunique())
print("Validation engines:", val_df["Engine_ID"].nunique())
print("Testing engines:", test_df["Engine_ID"].nunique())


Training engines: 70
Validation engines: 15
Testing engines: 15


In [8]:
FEATURES = [
    "Setting_1",
    "Setting_2",
    "Sensor_2",
    "Sensor_3",
    "Sensor_4",
    "Sensor_6",
    "Sensor_7",
    "Sensor_8",
    "Sensor_9",
    "Sensor_11",
    "Sensor_12",
    "Sensor_13",
    "Sensor_14",
    "Sensor_15",
    "Sensor_17",
    "Sensor_20",
    "Sensor_21"
]

TARGET = "RUL"


In [9]:
WINDOW = 30

def create_sequences(data, features, target, window=30):
    X = []
    y = []

    for engine_id, engine_data in data.groupby("Engine_ID"):

        engine_data = engine_data.sort_values("Cycle")

        feature_values = engine_data[features].values.astype("float32")
        target_values = engine_data[target].values.astype("float32")

        for i in range(window, len(engine_data)):
            X.append(feature_values[i-window:i])
            y.append(target_values[i])

    return np.array(X), np.array(y)


In [10]:
X_train, y_train = create_sequences(train_df, FEATURES, TARGET, WINDOW)
X_val, y_val = create_sequences(val_df, FEATURES, TARGET, WINDOW)
X_test, y_test = create_sequences(test_df, FEATURES, TARGET, WINDOW)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)


(12216, 30, 17)
(2720, 30, 17)
(2695, 30, 17)


## Train the point-prediction model

Same BiLSTM architecture as the other two notebooks.

In [11]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    Input,
    LSTM,
    Bidirectional,
    Dense,
    Dropout
)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau

tf.random.set_seed(42)

bilstm = Sequential([

    Input(shape=(WINDOW, len(FEATURES))),

    Bidirectional(
        LSTM(
            64,
            return_sequences=True
        )
    ),

    Dropout(0.2),

    Bidirectional(
        LSTM(32)
    ),

    Dropout(0.2),

    Dense(
        32,
        activation="relu"
    ),

    Dense(1)
])

bilstm.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=0.001
    ),
    loss="mse",
    metrics=["mae"]
)

bilstm.summary()


C:\Users\Shivam\AppData\Roaming\Python\Python310\site-packages\google\api_core\_python_version_support.py:273: FutureWarning: You are using a Python version (3.10.2) which Google will stop supporting in new releases of google.api_core once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.api_core past that date.
  warnings.warn(message, FutureWarning)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ bidirectional (Bidirectional)   │ (None, 30, 128)        │        41,984 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 30, 128)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_1 (Bidirectional) │ (None, 64)             │        41,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 85,313 (333.25 KB)

 Trainable params: 85,313 (333.25 KB)

 Non-trainable params: 0 (0.00 B)

In [12]:
early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

reduce_lr = ReduceLROnPlateau(
    monitor="val_loss",
    factor=0.5,
    patience=5,
    min_lr=1e-6
)

history_bilstm = bilstm.fit(
    X_train,
    y_train,

    validation_data=(
        X_val,
        y_val
    ),

    epochs=100,
    batch_size=64,

    shuffle=False,

    callbacks=[
        early_stopping,
        reduce_lr
    ],

    verbose=1
)


Epoch 1/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 11s 28ms/step - loss: 4454.4971 - mae: 56.4458 - val_loss: 1975.7677 - val_mae: 39.8377 - learning_rate: 0.0010
Epoch 2/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 1803.9015 - mae: 37.7496 - val_loss: 1748.6732 - val_mae: 37.1877 - learning_rate: 0.0010
Epoch 3/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 1775.4539 - mae: 37.3265 - val_loss: 1747.3157 - val_mae: 37.1479 - learning_rate: 0.0010
Epoch 4/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 1766.5442 - mae: 37.1908 - val_loss: 1737.3151 - val_mae: 37.0509 - learning_rate: 0.0010
Epoch 5/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 4s 23ms/step - loss: 1172.1830 - mae: 29.1477 - val_loss: 1604.6482 - val_mae: 34.1987 - learning_rate: 0.0010
Epoch 6/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 24ms/step - loss: 951.8423 - mae: 26.3420 - val_loss: 619.6361 - val_mae: 20.0649 - learning_rate: 0.0010
Epoch 7/100
191/191 ━━━━━━━━━━━━━━━━━━━━ 5s 23ms/step - loss: 597.3337 - mae: 19.5937 - v

## Per-point difficulty signal (MC Dropout)

Just the mean/std of many stochastic forward passes, same function as `dropout.ipynb`. Here it's not the final uncertainty estimate -- it's the raw material conformal calibration will turn into a *guaranteed* interval below.

In [13]:
def mc_dropout_predict(model, X, n_passes=100):

    predictions = []

    for _ in range(n_passes):

        pred = model(
            X,
            training=True
        ).numpy().ravel()

        predictions.append(pred)

    predictions = np.array(predictions)

    mean_pred = predictions.mean(axis=0)
    std_pred = predictions.std(axis=0)

    return mean_pred, std_pred


## Calibrate on `val_df`

This is the one non-negotiable rule of conformal prediction: the calibration set must be data the model never trained on **and** that you don't also report final accuracy on. `val_df` satisfies both -- it's untouched by training, and it's a different split from the official test set and from `X_test` (the held-out-split evaluation set). If you calibrate on the same data you evaluate on, the coverage guarantee below is not actually a guarantee anymore.

`ALPHA = 0.10` targets 90% coverage -- i.e. we want the true RUL to fall inside the predicted interval for roughly 90% of engines. This is conformal prediction's own internal target and is a separate knob from `ACCEPT_RATE` (which governs the accept/abstain sweep, same as the other two notebooks).

In [14]:
val_mean, val_std = mc_dropout_predict(
    bilstm,
    X_val,
    n_passes=100
)

EPS = 1e-6

nonconformity = np.abs(y_val - val_mean) / (val_std + EPS)

ALPHA = 0.10  # target 90% coverage

q = np.quantile(nonconformity, 1 - ALPHA)

print("Conformal quantile (q):", q)


Conformal quantile (q): 5.212575


Interval half-width for any new point `x` is `q * sigma(x)` -- narrow where the model's own MC-Dropout uncertainty is low, wide where it's high, but scaled so that overall coverage on unseen data matches the 90% target. That per-point half-width is what gets thresholded in the accept/abstain sweep below, the same way `dropout.ipynb` thresholds raw MC-Dropout std and `deep_ensembles.ipynb` thresholds ensemble std.

In [15]:
heldout_mean, heldout_std = mc_dropout_predict(
    bilstm,
    X_test,
    n_passes=100
)

heldout_interval_width = q * heldout_std

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

print("Held-out split:")
print("MAE :", mean_absolute_error(y_test, heldout_mean))
print("RMSE:", np.sqrt(mean_squared_error(y_test, heldout_mean)))
print("R\u00b2  :", r2_score(y_test, heldout_mean))

coverage = (
    np.abs(y_test - heldout_mean) <= heldout_interval_width
).mean()

print(f"\nEmpirical coverage on held-out split: {coverage:.1%} (target: {1 - ALPHA:.0%})")


Held-out split:
MAE : 16.760194778442383
RMSE: 21.163159097320882
R²  : 0.7436487674713135

Empirical coverage on held-out split: 88.2% (target: 90%)


## Accept/abstain threshold, calibrated on `val_df`

In [16]:
ACCEPT_RATE = 0.80

val_interval_width = q * val_std

cp_threshold = np.quantile(
    val_interval_width,
    ACCEPT_RATE
)

print("Conformal interval-width threshold:", cp_threshold)


Conformal interval-width threshold: 40.008236


## Official test set (literature-comparable)

In [17]:
RUL_CAP = 125

test_df_final = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\Dataset\processed_test.csv")
output_df = pd.read_csv(r"C:\Users\Shivam\Desktop\Capstone_updated\Dataset\RUL_FD001.csv")
output_df["Engine_ID"] = np.arange(1, len(output_df) + 1)
output_df["RUL"] = output_df["RUL"].clip(upper=RUL_CAP)

def create_test_sequences(test_df, features, window=30):
    X, engine_ids = [], []
    for engine_id, engine_data in test_df.groupby("Engine_ID"):
        engine_data = engine_data.sort_values("Cycle")
        values = engine_data[features].values.astype("float32")
        if len(values) >= window:
            X.append(values[-window:])
            engine_ids.append(engine_id)
    return np.array(X), np.array(engine_ids)

X_test_final, test_engine_ids = create_test_sequences(test_df_final, FEATURES, WINDOW)


In [18]:
cp_test_mean, cp_test_std = mc_dropout_predict(
    bilstm,
    X_test_final,
    n_passes=100
)

cp_test_interval_width = q * cp_test_std

pred_df = pd.DataFrame({
    "Engine_ID": test_engine_ids,
    "Predicted_RUL": cp_test_mean,
    "IntervalWidth": cp_test_interval_width
})

comparison = pred_df.merge(output_df, on="Engine_ID")

actual = comparison["RUL"].values
cp_test_mean = comparison["Predicted_RUL"].values
cp_test_interval_width = comparison["IntervalWidth"].values


In [19]:
def nasa_score(actual, predicted):
    """NASA / PHM08 asymmetric RUL scoring function (Saxena et al., 2008).
    Penalizes late predictions (predicted RUL > actual RUL) more heavily
    than early ones, since a late prediction means maintenance would be
    scheduled after the engine could already have failed.
    """
    d = np.asarray(predicted) - np.asarray(actual)
    s = np.where(d < 0, np.exp(-d / 13) - 1, np.exp(d / 10) - 1)
    return np.sum(s)


In [20]:
print("100% coverage:")
print("MAE  :", mean_absolute_error(actual, cp_test_mean))
print("RMSE :", np.sqrt(mean_squared_error(actual, cp_test_mean)))
print("R\u00b2   :", r2_score(actual, cp_test_mean))
print("Score:", nasa_score(actual, cp_test_mean))

official_coverage = (
    np.abs(actual - cp_test_mean) <= cp_test_interval_width
).mean()

print(f"\nEmpirical coverage on official test set: {official_coverage:.1%} (target: {1 - ALPHA:.0%})")


100% coverage:
MAE  : 13.86269760131836
RMSE : 19.079854900840033
R²   : 0.7733057737350464
Score: 1138.9748816182152

Empirical coverage on official test set: 89.0% (target: 90%)


## Accept / abstain sweep

In [21]:
accept = cp_test_interval_width <= cp_threshold

print("Accepted :", accept.sum())
print("Abstained:", (~accept).sum())
print("Coverage :", accept.mean())


Accepted : 85
Abstained: 15
Coverage : 0.85


In [22]:
y_true_accept = actual[accept]
y_pred_accept = cp_test_mean[accept]

mae = mean_absolute_error(y_true_accept, y_pred_accept)
rmse = np.sqrt(mean_squared_error(y_true_accept, y_pred_accept))
r2 = r2_score(y_true_accept, y_pred_accept)
score = nasa_score(y_true_accept, y_pred_accept)

print("Accepted predictions:")
print("MAE  :", mae)
print("RMSE :", rmse)
print("R\u00b2   :", r2)
print("Score:", score)


Accepted predictions:
MAE  : 14.123754501342773
RMSE : 19.254825135939953
R²   : 0.768051028251648
Score: 932.2819536417885


In [23]:
for rate in [0.9, 0.8, 0.7]:

    threshold = np.quantile(
        val_interval_width,
        rate
    )

    accept = cp_test_interval_width <= threshold

    rmse = np.sqrt(
        mean_squared_error(
            actual[accept],
            cp_test_mean[accept]
        )
    )

    score = nasa_score(
        actual[accept],
        cp_test_mean[accept]
    )

    print(
        f"Acceptance={rate:.0%} | "
        f"Accepted={accept.sum()} | "
        f"RMSE={rmse:.3f} | "
        f"Score={score:.1f}"
    )


Acceptance=90% | Accepted=95 | RMSE=18.454 | Score=945.2
Acceptance=80% | Accepted=85 | RMSE=19.255 | Score=932.3
Acceptance=70% | Accepted=75 | RMSE=18.476 | Score=766.8


In [24]:
import os

METHOD = "conformal"
SCENARIO = "clean_capped"

RESULTS_DIR = r"C:\Users\Shivam\Desktop\Capstone_updated\results"
os.makedirs(RESULTS_DIR, exist_ok=True)

predicted = cp_test_mean
uncertainty = cp_test_interval_width

results_df = pd.DataFrame({
    "Engine_ID": test_engine_ids,
    "Actual_RUL": actual,
    "Predicted_RUL": predicted,
    "Uncertainty": uncertainty
})

out_path = os.path.join(
    RESULTS_DIR,
    f"{METHOD}__{SCENARIO}.csv"
)

results_df.to_csv(out_path, index=False)

print(f"Saved {len(results_df)} rows -> {out_path}")

Saved 100 rows -> C:\Users\Shivam\Desktop\Capstone_updated\results\conformal__clean_capped.csv
